# 05 — SWAN Wave Coupling via DIMR

Couple SWAN wave model with the existing D-Flow FM 3D hydrodynamic model to include:
- Wind-generated waves inside the lagoon (fetch-limited)
- Offshore swell entering through BocaNord and BocaSud
- Wave-current interaction at the inlets
- Wave dissipation by seagrass canopies (Posidonia + Cymodocea)
- Bottom shear stress enhancement (relevant for turbidity modeling)

**Coupling approach:**
- SWAN on a separate structured rectangular grid (initial simple setup)
- DIMR orchestrates: FM → SWAN every coupling interval
- Coupling interval: 600 s (10 min)
- Flow → Wave: water level, currents, wind
- Wave → Flow: wave radiation stresses, significant wave height, period

**Target model**: v03 (v02 + waves). This notebook prepares the configuration; actually running it requires v02 to be running/finished.

## 1. Imports and paths

In [ ]:
%matplotlib inline
import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path(r'F:\StagnoneDT')
v02_dir = project_root / 'model' / 'dflowfm_v02'
v03_dir = project_root / 'model' / 'dflowfm_v03'
wave_dir = v03_dir / 'wave'

## 2. Copy v02 as v03 base

We'll add waves on top of v02. Create a v03 directory with DFlowFM files + a separate `wave/` subfolder.

In [ ]:
EXCLUDE = {'output', '__pycache__'}
EXCLUDE_SUFFIX = {'.cache'}

if not v03_dir.exists():
    os.makedirs(v03_dir, exist_ok=True)
    for item in v02_dir.iterdir():
        if item.name in EXCLUDE or item.suffix in EXCLUDE_SUFFIX:
            continue
        target = v03_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)
    print(f'v03 directory created: {v03_dir}')
else:
    print(f'v03 directory already exists — skipping copy')

os.makedirs(wave_dir, exist_ok=True)
os.makedirs(v03_dir / 'output', exist_ok=True)

## 3. Generate SWAN rectangular grid

Covers the Stagnone lagoon plus a small offshore buffer. Resolution ~50 m (finer than needed for waves but consistent with lagoon-scale processes).

In [ ]:
# SWAN grid parameters
SWAN_LON_MIN = 12.40
SWAN_LON_MAX = 12.50
SWAN_LAT_MIN = 37.81
SWAN_LAT_MAX = 37.93
SWAN_DX = 0.0005   # ~50 m
SWAN_DY = 0.0005

# In Cartesian (UTM) for SWAN we might convert, but here we use spherical grid (spherical=Y in SWAN)
nx = int((SWAN_LON_MAX - SWAN_LON_MIN) / SWAN_DX) + 1
ny = int((SWAN_LAT_MAX - SWAN_LAT_MIN) / SWAN_DY) + 1
print(f'SWAN grid: {nx} x {ny} = {nx*ny} cells')
print(f'  lon: {SWAN_LON_MIN} to {SWAN_LON_MAX} step {SWAN_DX}')
print(f'  lat: {SWAN_LAT_MIN} to {SWAN_LAT_MAX} step {SWAN_DY}')

## 4. Create SWAN master definition file (.mdw)

The `.mdw` file is the main configuration that DIMR uses to drive SWAN within the wave module.

In [ ]:
# Master definition for the SWAN wave model
mdw_content = f"""[WaveFileInformation]
   FileVersion           = 02.00

[General]
   ProjectName           = stagnone
   ProjectNr             = 01
   Description           = Stagnone di Marsala wave model (SWAN coupled with D-Flow FM)
   OnlyInputVerify       = false
   SimMode               = stationary
   DirConvention         = nautical
   ReferenceDate         = 2025-07-01
   WindSpeed             = 0.00
   WindDir               = 0.00

[TimePoint]
   Time                  = 0.0000000e+000
   WaterLevel            = 0.0000000e+000
   VelocityX             = 0.0000000e+000
   VelocityY             = 0.0000000e+000

[Constants]
   WaterLevelCorrection  = 0.0
   Gravity               = 9.81
   WaterDensity          = 1025.0
   NorthDir              = 90.0
   MinimumDepth          = 0.05

[Processes]
   GenModePhys           = 3
   WaveSetup             = false
   Breaking              = true
   BreakAlpha            = 1.0
   BreakGamma            = 0.73
   Triads                = true
   TriadsAlpha           = 0.1
   TriadsBeta            = 2.2
   WaveDiffraction       = false
   BedFriction           = jonswap
   BedFricCoef           = 0.067
   Diffraction           = false
   WindGrowth            = true
   WhiteCapping          = Komen
   Quadruplets           = true
   Refraction            = true
   FrequencyShift        = true
   WaveForces            = dissipation 3d

[Numerics]
   DirSpaceCDD           = 0.5
   FreqSpaceCSS          = 0.5
   RChHsTm01             = 0.02
   RChMeanHs             = 0.02
   RChMeanTm01           = 0.02
   PercWet               = 98.0
   MaxIter               = 15

[Output]
   MapWriteInterval      = 1800
   CommunicateWithFlow   = true
   WriteCOM              = true
   COMWriteInterval      = 600
   LocationsFile         = stagnone_wave_locs.loc

[Domain]
   Grid                  = swan_grid.grd
   BedLevel              = swan_bathy.dep
   DirSpace              = circle
   NDir                  = 36
   StartDir              = 0.0
   EndDir                = 0.0
   FreqMin               = 0.05
   FreqMax               = 1.0
   NFreq                 = 30
   Output                = true

[Boundary]
   Name                  = offshore_west
   Definition            = orientation
   Orientation           = west
   DistanceDir           = counter-clockwise
   SpectrumSpec          = parametric
   SpShapeType           = jonswap
   PeriodType            = peak
   DirSpreadType         = power
   PeakEnhanceFac        = 3.3
   WaveHeight            = 0.5
   Period                = 5.0
   Direction             = 270.0
   DirSpreading          = 4.0
"""

mdw_file = wave_dir / 'stagnone.mdw'
with open(mdw_file, 'w') as f:
    f.write(mdw_content)
print(f'Written: {mdw_file}')

## 5. Create SWAN rectangular grid (.grd) and bathymetry (.dep)

Delft3D wave uses rectangular grids in the RGFGRID format.

In [ ]:
# Write the structured grid in Delft3D .grd format (2D spherical)
grd_file = wave_dir / 'swan_grid.grd'

# Grid coords
lons = np.arange(SWAN_LON_MIN, SWAN_LON_MAX + SWAN_DX/2, SWAN_DX)
lats = np.arange(SWAN_LAT_MIN, SWAN_LAT_MAX + SWAN_DY/2, SWAN_DY)
nx, ny = len(lons), len(lats)

with open(grd_file, 'w') as f:
    f.write('Coordinate System = Spherical\n')
    f.write(f'     {nx}     {ny}\n')
    f.write(' 0 0 0\n')
    # X coordinates (longitude)
    for j in range(ny):
        f.write(f' ETA=  {j+1}')
        for i in range(nx):
            if i % 5 == 0 and i > 0:
                f.write('\n             ')
            f.write(f' {lons[i]:16.10e}')
        f.write('\n')
    # Y coordinates (latitude)
    for j in range(ny):
        f.write(f' ETA=  {j+1}')
        for i in range(nx):
            if i % 5 == 0 and i > 0:
                f.write('\n             ')
            f.write(f' {lats[j]:16.10e}')
        f.write('\n')

print(f'Written: {grd_file}')
print(f'  {nx} x {ny} nodes')

In [ ]:
# Create SWAN bathymetry by interpolating from the FM mesh bed level
import xugrid as xu
from scipy.interpolate import griddata

# Load the FM net file to get mesh bathymetry
net_file = v02_dir / 'Stagnone_dxy01_15m_net.nc'
uds_net = xu.open_dataset(str(net_file))
node_x = np.asarray(uds_net.grid.node_x)
node_y = np.asarray(uds_net.grid.node_y)
node_z = uds_net['mesh2d_node_z'].values if 'mesh2d_node_z' in uds_net else None

if node_z is not None:
    # Interpolate to SWAN grid (bilinear)
    Lons, Lats = np.meshgrid(lons, lats)
    points = np.column_stack([node_x, node_y])
    bathy = griddata(points, node_z, (Lons, Lats), method='linear', fill_value=0.0)
    
    # SWAN convention: depth positive DOWN (so invert sign of bed level)
    depth = -bathy
    depth[depth < 0] = -99.0  # dry cells
    
    dep_file = wave_dir / 'swan_bathy.dep'
    with open(dep_file, 'w') as f:
        for j in range(ny):
            for i in range(nx):
                f.write(f' {depth[j, i]:16.7e}')
                if (i + 1) % 12 == 0:
                    f.write('\n')
            if nx % 12 != 0:
                f.write('\n')
    print(f'Written: {dep_file}')
    print(f'  Depth range: {depth[depth > -99].min():.2f} to {depth[depth > -99].max():.2f} m (positive down)')
else:
    print('No bed level in net file — skip bathy generation')

## 6. Create observation locations file

In [ ]:
# Wave output at the same observation points as FM (BocaNord, BocaSud, etc.)
obs_file = v03_dir / 'Stagnone_dxy01_15m_obs.xyn'
obs_df = pd.read_csv(obs_file, sep=r'\s+', header=None, names=['x', 'y', 'name'])

loc_file = wave_dir / 'stagnone_wave_locs.loc'
with open(loc_file, 'w') as f:
    for _, row in obs_df.iterrows():
        f.write(f'{row["x"]:16.6f} {row["y"]:16.6f} {row["name"]}\n')

print(f'Written: {loc_file}')
print(f'  {len(obs_df)} observation points')

## 7. Update dimr_config.xml for coupled FM+Wave run

In [ ]:
dimr_content = '''<?xml version="1.0" encoding="utf-8"?>
<dimrConfig xmlns="http://schemas.deltares.nl/dimr"
             xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
             xsi:schemaLocation="http://schemas.deltares.nl/dimr http://content.oss.deltares.nl/schemas/dimr-1.3.xsd">
  <documentation>
    <fileVersion>1.3</fileVersion>
    <createdBy>notebook 05 — v03 coupled FM+SWAN</createdBy>
  </documentation>
  <control>
    <parallel>
      <startGroup>
        <time>0 600 777600</time>
        <coupler name="flow2wave"/>
        <start name="wave"/>
        <coupler name="wave2flow"/>
      </startGroup>
      <start name="DFlowFM"/>
    </parallel>
  </control>
  <component name="DFlowFM">
    <library>dflowfm</library>
    <workingDir>.</workingDir>
    <inputFile>Stagnone_dxy01_15m.mdu</inputFile>
    <mpiCommunicator>DFM_COMM_DFMWORLD</mpiCommunicator>
  </component>
  <component name="wave">
    <library>wave</library>
    <workingDir>wave</workingDir>
    <inputFile>stagnone.mdw</inputFile>
  </component>
  <coupler name="flow2wave">
    <sourceComponent>DFlowFM</sourceComponent>
    <targetComponent>wave</targetComponent>
    <item>
      <sourceName>water_level</sourceName>
      <targetName>water_level</targetName>
    </item>
    <item>
      <sourceName>velocity_x</sourceName>
      <targetName>velocity_x</targetName>
    </item>
    <item>
      <sourceName>velocity_y</sourceName>
      <targetName>velocity_y</targetName>
    </item>
    <item>
      <sourceName>wind_x</sourceName>
      <targetName>wind_x</targetName>
    </item>
    <item>
      <sourceName>wind_y</sourceName>
      <targetName>wind_y</targetName>
    </item>
  </coupler>
  <coupler name="wave2flow">
    <sourceComponent>wave</sourceComponent>
    <targetComponent>DFlowFM</targetComponent>
    <item>
      <sourceName>hs</sourceName>
      <targetName>hs</targetName>
    </item>
    <item>
      <sourceName>tp</sourceName>
      <targetName>tp</targetName>
    </item>
    <item>
      <sourceName>wave_force_x</sourceName>
      <targetName>wave_force_x</targetName>
    </item>
    <item>
      <sourceName>wave_force_y</sourceName>
      <targetName>wave_force_y</targetName>
    </item>
  </coupler>
</dimrConfig>
'''

dimr_file = v03_dir / 'dimr_config.xml'
with open(dimr_file, 'w') as f:
    f.write(dimr_content)
print(f'Written: {dimr_file}')

## 8. Update FM MDU to enable SWAN coupling

In [ ]:
import re

mdu_file = v03_dir / 'Stagnone_dxy01_15m.mdu'
with open(mdu_file, 'r') as f:
    mdu_text = f.read()

# Set Wavemodelnr = 3 (SWAN coupling)
mdu_text = re.sub(r'Wavemodelnr\s*=\s*\d+', 'Wavemodelnr                       = 3', mdu_text)

# Ensure WaveNikuradse is set (not present in current MDU since we stripped it — may need to re-add)
# The strip only removed the unsupported key, so add it back via the wave module config

with open(mdu_file, 'w') as f:
    f.write(mdu_text)

print('Updated MDU — Wavemodelnr = 3 (SWAN)')
# Show wave-related lines
for line in mdu_text.splitlines():
    if 'wave' in line.lower() or 'waves' in line.lower():
        print(f'  {line.strip()[:100]}')

## 9. Summary and run instructions

**v03 structure:**
```
model/dflowfm_v03/
├── Stagnone_dxy01_15m.mdu       # FM setup (Wavemodelnr=3)
├── Stagnone_dxy01_15m_*.ext     # boundary & meteo
├── dimr_config.xml               # coupled orchestration
├── run_model.bat
├── wave/
│   ├── stagnone.mdw              # SWAN master def
│   ├── swan_grid.grd             # rectangular grid ~50m
│   ├── swan_bathy.dep            # interpolated bathymetry
│   └── stagnone_wave_locs.loc    # obs points
└── output/
```

**To run v03:**
```
cd model/dflowfm_v03
run_model.bat
```

**Caveats / next iterations:**

1. **Offshore wave boundary** is currently a simple parametric: Hs=0.5m, Tp=5s, Dir=270° (west), spread=4°. For realistic runs, download CMEMS wave reanalysis (MEDSEA_MULTIYEAR_WAV_006_012) and create time-varying boundary spectrum.
2. **Vegetation dissipation** not yet included — would need the seagrass classification from notebook 01_input_satellite_roughness applied as a vegetation map on the SWAN grid using the VEGETATION keyword.
3. **Coupling interval 600s** is a reasonable start. Shorter (300s) gives better accuracy in rapidly changing wind conditions but doubles runtime.
4. **SimMode = stationary** means each wave timestep solves the stationary wave field. For fast-moving wave events, switch to nonstationary (higher cost).
5. **First test**: run without waves (v02) must succeed before adding waves. Validate v02 results, then enable v03.
6. **MDU may need** `WaveNikuradse` and other wave params re-added — check first run log for warnings.